# ✅ Soluciones — Clase 21: Gradient Boosting

Soluciones comentadas de los ejercicios de la Clase 21.

> Usa este archivo solo después de intentar los ejercicios por tu cuenta.

## Solución Ejercicio 1 — GradientBoostingClassifier básico

In [ ]:
# Solución E1: entrenar GBM y comparar train vs test
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../datasets/estudiantes.csv")
df["estado"] = ((df["evaluacion_python"] >= 60) & (df["asistencia_pct"] >= 70)).astype(int)

X = df[["asistencia_pct", "evaluacion_python", "evaluacion_pandas", "edad"]]
y = df["estado"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

print(f"Train: {gbm.score(X_train, y_train):.3f}  |  Test: {gbm.score(X_test, y_test):.3f}")
print("Diferencia vs Random Forest: los árboles se construyen en serie, cada uno corrige errores del anterior.")
print("Random Forest construye todos los árboles en paralelo, independientemente.")

## Solución Ejercicio 2 — Efecto del learning_rate

In [ ]:
# Solución E2: comparar tres learning_rate
for lr in [0.01, 0.1, 0.5]:
    m = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    m.fit(X_train, y_train)
    print(f"learning_rate={lr:.2f} → Test accuracy: {m.score(X_test, y_test):.3f}")

print("\n→ lr=0.01: aprende demasiado despacio con solo 100 árboles (necesita más).")
print("→ lr=0.5:  puede sobreajustar al tomar pasos demasiado grandes.")
print("→ lr=0.1:  buen equilibrio habitual.")

## Solución Ejercicio 3 — Comparación de 4 modelos

In [ ]:
# Solución E3: tabla comparativa de accuracy en test
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Árbol (d=3)":         DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

print(f"{'Modelo':<25} {'Test Accuracy':>14}")
print("-" * 42)
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    print(f"{nombre:<25} {modelo.score(X_test, y_test):>14.3f}")

print("\n→ Para simplificar: eliminamos el árbol individual primero (menor precisión y no gana nada vs GBM).")

## Solución Ejercicio 4 — XGBoost

In [ ]:
# Solución E4: XGBClassifier con mismos parámetros
try:
    import xgboost as xgb
    xgb_m = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                                random_state=42, eval_metric="logloss", verbosity=0)
    xgb_m.fit(X_train, y_train)
    print(f"XGBoost test accuracy: {xgb_m.score(X_test, y_test):.3f}")
    print("Diferencia principal: XGBoost es más rápido y tiene regularización L1/L2 incorporada.")
except ImportError:
    print("XGBoost no instalado. Ejecuta: pip install xgboost")
    print("Diferencia: XGBoost es más rápido, tiene regularización L1/L2 y maneja nulos nativamente.")

## Solución Ejercicio 5 — Reflexión final

In [ ]:
# Solución E5: reflexión en texto
reflexion = """
¿Cuándo usar GBM en lugar de Regresión Logística?
→ Cuando la relación entre las variables no es lineal y necesitamos mayor precisión.
→ La Regresión Logística asume relaciones lineales; GBM puede capturar patrones mucho más complejos.
→ Sin embargo, para un modelo que debe explicarse a no-técnicos, la Regresión Logística sigue siendo útil.

¿Qué hacer si Train=0.99 y Test=0.65?
→ Es sobreajuste claro. Las estrategias son:
   1. Reducir n_estimators o max_depth.
   2. Reducir learning_rate (y aumentar n_estimators para compensar).
   3. Aplicar regularización (si uso XGBoost, usar reg_alpha o reg_lambda).
   4. Agregar más datos de entrenamiento si es posible.
"""
print(reflexion)